# Ü-I — Iceberg-Tabelle in Glue erzeugen & im Catalog anlegen

Ziel: `raw.orders` als **Iceberg**-Tabelle `processed.orders_iceberg` schreiben,
sodass sie im **Glue Data Catalog** steht und aus Athena abfragbar ist.

Iceberg ist in Glue 5.0 nativ — aktiviert über `--datalake-formats=iceberg` plus
die `--conf`-Kette im `%%configure`-Block unten (named Catalog `glue_catalog`).

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
# --datalake-formats=iceberg lädt die Iceberg-Jars; die --conf-Kette verdrahtet
# den named Catalog `glue_catalog` mit dem Glue Data Catalog. Muss beim
# Session-Bau stehen (spark.sql.extensions lässt sich später nicht mehr setzen).
%%configure
{
  "--datalake-formats": "iceberg",
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://gfu-glue-training-629452195361/processed/ --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO"
}

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

## 1) Iceberg-Tabelle anlegen (Struktur `<catalog>.<datenbank>.<tabelle>`)

Die Zelle unten ist vorgefertigt — ersetze nur die `____`. Zwei Fragen:

- `____catalog`: Über welchen Catalog schreibst du, damit die Tabelle sofort im
  Data Catalog steht? (Den Namen hast du oben im `%%configure` selbst gesetzt.)
- `____format`: Welches Wort hinter `USING` macht daraus eine Iceberg-Tabelle?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____catalog : welcher named Catalog schreibt den Eintrag sofort in den Data Catalog?
#   ____format  : welches Format macht es zur Iceberg-Tabelle?
spark.sql("""
    CREATE TABLE IF NOT EXISTS ____catalog.processed.orders_iceberg (
        customer_id string,
        order_total double,
        order_date  string,
        status      string
    )
    USING ____format
    LOCATION 's3://gfu-glue-training-629452195361/processed/orders_iceberg'
    TBLPROPERTIES ('format-version' = '2')
""")

## 2) Aus `raw.orders` befüllen

`INSERT INTO … SELECT …` ist vorgefertigt. Ersetze die `____`.

**Backticks (`` ` ``):** Einige Spalten in `raw.orders` haben ein Leerzeichen im
Namen (Struktur `` `<wort> <wort>` ``). Ohne Backticks liest Spark so einen Namen als
zwei getrennte Spalten und meldet einen Fehler. Backticks fassen alles dazwischen zu
**einem Namen** zusammen. Nötig bei Leerzeichen/Sonderzeichen, nicht bei
Ein-Wort-Namen. (Achtung: `'...'` = Text-Wert, `` `...` `` = Spaltenname.)

Spalten, die als Text vorliegen, castest du bei Bedarf mit `CAST(... AS <typ>)`;
leere Zellen werden dabei zu NULL.

In [ ]:
# Rohspalten heißen `cust id`, `order total`, `order date` — MIT Leerzeichen.
#   ____col  : Rohspaltenname richtig quoten (Backticks!), damit Spark ihn findet.
#   ____cast : `order total` ist Text → nach double casten (leere Zelle wird NULL).
#   ____src  : Quelltabelle im Default-Catalog.
spark.sql("""
    INSERT INTO glue_catalog.processed.orders_iceberg
    SELECT
        ____col                       AS customer_id,
        CAST(____cast AS double)      AS order_total,
        ____col                       AS order_date,
        status                        AS status
    FROM ____src
""")

## 3) Prüfen

Zwei `____` füllen:

- `____tbl`: der volle Tabellenname in der Struktur `<catalog>.<datenbank>.<tabelle>`
  für den Row-Count.
- `____meta`: das Iceberg-Suffix, das die Snapshots (die Versions-Historie der
  Tabelle) auflistet.

`DESCRIBE EXTENDED` sollte `Provider: iceberg` zeigen.

In [ ]:
# ____tbl  : voll qualifizierte Iceberg-Tabelle.
# ____meta : Iceberg-Metadaten-Suffix, das die Snapshots als Zeilen liefert.
print("rows:", spark.sql("SELECT count(*) AS n FROM ____tbl").collect()[0]["n"])
spark.sql("DESCRIBE EXTENDED glue_catalog.processed.orders_iceberg").show(100, truncate=False)  # Provider: iceberg
spark.sql("SELECT snapshot_id, operation, summary FROM glue_catalog.processed.orders_iceberg.____meta").show(truncate=False)

## 4) Gegencheck in Athena (als Trainee)

Beweist, dass die Tabelle wirklich im **Glue Data Catalog** steht — Athena v3 liest
Iceberg über den Catalog nativ, ohne Extra-Setup. In der Athena-Konsole (Datenbank
`processed`) ausführen:

```sql
SELECT count(*) FROM processed.orders_iceberg;

SELECT status, count(*) AS n
FROM processed.orders_iceberg
GROUP BY status
ORDER BY n DESC;
```

**Aufräumen (Spark-Zelle, nur wenn gewünscht):**
`spark.sql("DROP TABLE glue_catalog.processed.orders_iceberg PURGE")` — löscht
Catalog-Eintrag **und** S3-Dateien.